Food Delivery Time Prediction Using XGBoost Regressor with Hyperparameter Tuning

### **Problem Statement**
Food delivery platforms face the challenge of accurately predicting
delivery time to improve customer satisfaction and operational efficiency.
Delivery time is influenced by multiple real-world factors such as
distance, traffic level, weather conditions, preparation time, vehicle
type and partner ratings.

This project uses XGBoost Regressor (Extreme Gradient Boosting) — the
most powerful and widely-used ML algorithm in industry and data science
competitions — to predict food delivery time (in minutes). XGBoost's
gradient boosting optimization, built-in regularization and speed make
it superior to traditional regression models for this real-world task.

### **Objective**
- Predict food delivery time (in minutes) using XGBoost Regressor

- Perform Feature Engineering to extract 4 new meaningful features
- Apply One-Hot Encoding on categorical features
- Follow correct ML pipeline — Encoding → Split → Model
- Improve model using GridSearchCV Hyperparameter Tuning
- Evaluate using MAE, MSE, RMSE and R² Score
- Compare Before and After tuning performance

### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### 2. Load Dataset

In [2]:
df = pd.read_csv("food_delivery_dataset.csv")

### 3. Data Understanding

In [3]:
df.head()

,Order_ID,Distance_km,Order_Time,Traffic_Level,Weather,Restaurant_Rating,Prep_Time_min,Delivery_Partner_Rating,Vehicle_Type,Delivery_Time_min
0,1001,4.06,21,Medium,Rainy,4.6,20,4.7,Bike,58
1,1002,9.53,17,High,Clear,3.4,25,4.3,Scooter,90
2,1003,7.45,12,Medium,Cloudy,3.7,26,3.9,Bike,78
3,1004,6.19,17,Low,Rainy,3.7,28,4.5,Bike,69
4,1005,1.98,14,Medium,Clear,5.0,27,4.4,Scooter,46


In [4]:
df.shape

(1000, 10)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Order_ID                 1000 non-null   int64  
 1   Distance_km              1000 non-null   float64
 2   Order_Time               1000 non-null   int64  
 3   Traffic_Level            1000 non-null   object 
 4   Weather                  1000 non-null   object 
 5   Restaurant_Rating        1000 non-null   float64
 6   Prep_Time_min            1000 non-null   int64  
 7   Delivery_Partner_Rating  1000 non-null   float64
 8   Vehicle_Type             1000 non-null   object 
 9   Delivery_Time_min        1000 non-null   int64  
dtypes: float64(3), int64(4), object(3)
memory usage: 78.3+ KB


In [6]:
df.describe()

,Order_ID,Distance_km,Order_Time,Restaurant_Rating,Prep_Time_min,Delivery_Partner_Rating,Delivery_Time_min
count,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000
mean,1500.500000,5.157360,16.15000,3.988600,24.892000,4.248100,66.614000
std,288.819436,2.775165,3.81314,0.576196,8.405771,0.431048,18.021537
min,1001.000000,0.540000,10.00000,3.000000,10.000000,3.500000,21.000000
25%,1250.750000,2.740000,13.00000,3.500000,18.000000,3.900000,53.000000
50%,1500.500000,5.220000,16.00000,4.000000,25.000000,4.200000,67.000000
75%,1750.250000,7.572500,19.00000,4.500000,32.000000,4.600000,80.000000
max,2000.000000,10.000000,22.00000,5.000000,39.000000,5.000000,119.000000


In [7]:
print(df.isnull().sum())

Order_ID                   0
Distance_km                0
Order_Time                 0
Traffic_Level              0
Weather                    0
Restaurant_Rating          0
Prep_Time_min              0
Delivery_Partner_Rating    0
Vehicle_Type               0
Delivery_Time_min          0
dtype: int64


### 4. Data Preprocessing

#### 4.1 Drop Irrelevant Column

In [8]:
df.drop('Order_ID', axis=1, inplace=True)

#### 4.2 Feature Engineering

In [9]:
df['Distance_x_Traffic'] = df['Distance_km'] * df['Traffic_Level'].map({'Low': 1, 'Medium': 2, 'High': 3})
df['Rating_Diff']        = df['Restaurant_Rating'] - df['Delivery_Partner_Rating']
df['Total_Time_Est']     = df['Distance_km'] * 5 + df['Prep_Time_min']
df['Rush_Hour']          = df['Order_Time'].apply(lambda x: 1 if x in range(18, 22) else 0)

#### 4.3 Encoding

In [10]:
df = pd.get_dummies(df, drop_first=True, dtype=int)


### 5. Model Building
#### 5.1 Split Features & Target

In [11]:
X = df.drop('Delivery_Time_min', axis=1)
y = df['Delivery_Time_min']

#### 5.2 Train-Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


#### 5.3 Model Before Tuning
- ####  Model Selection

In [13]:
model_before = XGBRegressor(random_state=42, verbosity=0)

- #### Train Model

In [14]:
model_before.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

- #### Make Predictions

In [15]:
y_pred_before = model_before.predict(X_test)

- #### Model Evaluation

In [16]:
print("BEFORE TUNING")
print("MAE  :", mean_absolute_error(y_test, y_pred_before))
print("MSE  :", mean_squared_error(y_test, y_pred_before))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_before)))
print("R²   :", r2_score(y_test, y_pred_before))

BEFORE TUNING
MAE  : 3.770045280456543
MSE  : 22.569435119628906
RMSE : 4.750729956504464
R²   : 0.9389935731887817


#### 5.4 Hyperparameter Tuning
- #### Define Hyperparameters

In [17]:
params = {
    'n_estimators'    : [50, 100, 200],
    'max_depth'       : [3, 5, 7],
    'learning_rate'   : [0.01, 0.1, 0.2],
    'subsample'       : [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

- ####  Apply GridSearchCV

In [18]:
grid = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    param_grid=params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

#### 5.5 Model After Tuning
- #### Train with GridSearchCV

In [19]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None...
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 1.0],
                         'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [3, 5, 7], 'n_estimators': [50, 100, 200],
                         'subsample': [0.8, 1.0]},
             scoring='r2')

- #### Build Best Model

In [20]:
best_model = grid.best_estimator_

- #### Make Predictions

In [21]:
y_pred_after = best_model.predict(X_test)

- #### Model Evaluation

In [22]:
print("AFTER TUNING")
print("Best Parameters :", grid.best_params_)
print("MAE  :", mean_absolute_error(y_test, y_pred_after))
print("MSE  :", mean_squared_error(y_test, y_pred_after))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_after)))
print("R²   :", r2_score(y_test, y_pred_after))

AFTER TUNING
Best Parameters : {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
MAE  : 3.4816253185272217
MSE  : 17.052345275878906
RMSE : 4.129448543798421
R²   : 0.9539065957069397


#### 5.6 Prediction Report

In [23]:
print(pd.DataFrame({
    'Actual'   : y_test.values,
    'Predicted': y_pred_after.round(2),
    'Error'    : (y_test.values - y_pred_after).round(2)
}).head(10))

   Actual   Predicted  Error
0      73   74.040001  -1.04
1     100  105.879997  -5.88
2      75   81.739998  -6.74
3      79   72.290001   6.71
4      86   84.470001   1.53
5     109  104.580002   4.42
6      70   74.110001  -4.11
7      94   95.010002  -1.01
8      44   50.529999  -6.53
9      74   76.489998  -2.49


### **Key Insights**
- R² improved from 93.89% → 95.39% after hyperparameter tuning 

- All 4 metrics improved significantly after tuning 
- MAE reduced from 3.770 → 3.482 (better prediction accuracy) 
- MSE reduced from 22.569 → 17.052 (significant reduction!) 
- RMSE reduced from 4.751 → 4.129 
- R² improvement of +1.50% — strong result for XGBoost 
- Best Parameters: colsample_bytree=0.8, learning_rate=0.1,
  max_depth=3, n_estimators=200, subsample=0.8
- colsample_bytree=0.8 → uses 80% features per tree (reduces overfitting)
- subsample=0.8 → uses 80% training samples per tree (better generalization)
- max_depth=3 → shallow trees with 200 estimators = strong ensemble
- Feature Engineering added 4 meaningful features:
  Distance_x_Traffic, Rating_Diff, Total_Time_Est, Rush_Hour
- Average prediction error = only 3.48 minutes — production ready! 
- XGBoost Regressor best performing model in delivery time portfolio:
  AdaBoost R²: 94.64% < XGBoost R²: 95.39% 


### **Conclusion**
The XGBoost Regressor successfully predicted food delivery time with
an outstanding R² Score of 95.39% after hyperparameter tuning —
a significant improvement from the 93.89% baseline (+1.50%).

Feature Engineering added powerful interaction features like
Distance_x_Traffic and Total_Time_Est that helped XGBoost capture
real-world delivery patterns more effectively. GridSearchCV found the
optimal combination of n_estimators=200, learning_rate=0.1, max_depth=3,
subsample=0.8 and colsample_bytree=0.8.

With an average prediction error of only 3.48 minutes, this model
is production-ready for real-world food delivery platforms. XGBoost
outperformed all previous regression models in this portfolio —
confirming its position as the go-to algorithm for tabular data
prediction tasks.